<a href="https://colab.research.google.com/github/Asish-san/Logistics-Data-Analyst/blob/main/Week_1_Task_Strategic_Planning_and_Data_Exploration_in_Logistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WEEK 1

In [10]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
from scipy.spatial.distance import cdist

# ==========================================
# 1. DATA GENERATION & CLEANING PIPELINE
# ==========================================
def load_and_clean_logistics_data(n_records=1000):
    """Simulates loading ERP/WMS logistics logs and performing preprocessing."""
    np.random.seed(42)
    dates = pd.date_range(start="2025-01-01", periods=n_records, freq="D")

    df = pd.DataFrame({
        "order_date": np.random.choice(dates, n_records),
        "sku_id": np.random.choice(["SKU_DAIRY_01", "SKU_PROD_02", "SKU_PHARM_03", "SKU_MEAT_04"], n_records),
        "rfc_id": np.random.choice(["RFC_NORTH", "RFC_SOUTH", "RFC_EAST"], n_records),
        "order_qty": np.random.poisson(lam=120, size=n_records).astype(float),
        "delivered_qty": np.random.poisson(lam=115, size=n_records).astype(float),
        "scheduled_lead_hrs": np.full(n_records, 24.0),
        "actual_lead_hrs": np.random.normal(loc=22.5, scale=4.5, size=n_records),
        "spoiled_units": np.random.binomial(n=20, p=0.15, size=n_records),
        "ambient_temp_c": np.random.uniform(10, 35, size=n_records)
    })

    # Introduce synthetic missing values and outliers
    df.loc[np.random.choice(df.index, 25), "actual_lead_hrs"] = np.nan
    df.loc[np.random.choice(df.index, 5), "order_qty"] = 5000.0  # Extreme outlier

    # Cleaning Step 1: Impute missing lead times by RFC median
    df["actual_lead_hrs"] = df.groupby("rfc_id")["actual_lead_hrs"].transform(
        lambda x: x.fillna(x.median())
    )

    # Cleaning Step 2: Cap outliers using IQR method
    Q1 = df["order_qty"].quantile(0.25)
    Q3 = df["order_qty"].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    df["order_qty"] = np.where(df["order_qty"] > upper_bound, upper_bound, df["order_qty"])

    # Feature Engineering: Binary flags for KPI calculation
    df["on_time"] = df["actual_lead_hrs"] <= df["scheduled_lead_hrs"]
    df["in_full"] = df["delivered_qty"] >= df["order_qty"]
    df["otif_flag"] = df["on_time"] & df["in_full"]

    return df.sort_values("order_date").reset_index(drop=True)

# ==========================================
# 2. EXPLORATORY DATA ANALYSIS & KPI ENGINE
# ==========================================
def compute_baseline_kpis(df):
    """Calculates core supply chain KPIs across Regional Fulfillment Centers."""
    kpi_summary = df.groupby("rfc_id").agg(
        total_orders=("order_date", "count"),
        otif_rate_pct=("otif_flag", lambda x: round(x.mean() * 100, 2)),
        avg_lead_time_hrs=("actual_lead_hrs", lambda x: round(x.mean(), 2)),
        spoilage_rate_pct=("spoiled_units", lambda x: round((x.sum() / df.loc[x.index, "order_qty"].sum()) * 100, 2))
    )
    print("--- BASELINE LOGISTICS KPI SUMMARY ---")
    print(kpi_summary)
    return kpi_summary

# ==========================================
# 3. CLUSTERING: SKU INVENTORY SEGMENTATION
# ==========================================
def segment_skus_kmeans(df):
    """Applies K-Means clustering to classify SKUs by volume, volatility, and spoilage risk."""
    sku_profile = df.groupby("sku_id").agg(
        mean_demand=("order_qty", "mean"),
        demand_cv=("order_qty", lambda x: x.std() / x.mean()),
        spoilage_ratio=("spoiled_units", lambda x: x.sum() / df.loc[x.index, "order_qty"].sum())
    )

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(sku_profile)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    sku_profile["cluster_tier"] = kmeans.fit_predict(scaled_features)
    return sku_profile

# ==========================================
# 4. REGRESSION: DEMAND FORECASTING MODEL
# ==========================================
def train_demand_forecast_model(df):
    """Trains a Gradient Boosting Regressor with lag features to predict SKU demand."""
    daily_demand = df.groupby("order_date").agg(
        total_qty=("order_qty", "sum"),
        avg_temp=("ambient_temp_c", "mean")
    ).reset_index()

    # Engineer time-series lag and rolling features
    daily_demand["day_of_week"] = daily_demand["order_date"].dt.dayofweek
    daily_demand["lag_1"] = daily_demand["total_qty"].shift(1)
    daily_demand["rolling_mean_7"] = daily_demand["total_qty"].rolling(window=7).mean()
    daily_demand = daily_demand.dropna().reset_index(drop=True)

    features = ["day_of_week", "avg_temp", "lag_1", "rolling_mean_7"]
    X = daily_demand[features]
    y = daily_demand["total_qty"]

    # Chronological Train/Test Split (80% train, 20% test)
    split_idx = int(len(daily_demand) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, predictions) * 100
    print(f"\nForecast Model Test MAPE: {mape:.2f}%")
    return model

# ==========================================
# 5. OPTIMIZATION: LAST-MILE ROUTE HEURISTIC
# ==========================================
def optimize_delivery_route(depot_coord, customer_coords, demands, truck_capacity=500):
    """
    Illustrates a Capacitated Nearest-Neighbor Routing Heuristic for last-mile delivery.
    Minimizes Euclidean transit distance while respecting reefer payload limits.
    """
    all_coords = np.vstack([depot_coord, customer_coords])
    dist_matrix = cdist(all_coords, all_coords, metric="euclidean")

    unvisited = set(range(1, len(all_coords)))
    routes = []

    while unvisited:
        current_node = 0  # Start at Depot
        current_load = 0
        route = [0]

        while unvisited:
            # Filter feasible next customers based on remaining truck capacity
            feasible = [node for node in unvisited if current_load + demands[node - 1] <= truck_capacity]
            if not feasible:
                break
            # Greedy step: select nearest feasible customer
            next_node = min(feasible, key=lambda node: dist_matrix[current_node, node])
            route.append(next_node)
            current_load += demands[next_node - 1]
            current_node = next_node
            unvisited.remove(next_node)

        route.append(0)  # Return to Depot
        routes.append((route, current_load))

    return routes

if __name__ == "__main__":
    # Execute End-to-End Strategic Prototype
    clean_df = load_and_clean_logistics_data()
    kpis = compute_baseline_kpis(clean_df)
    sku_clusters = segment_skus_kmeans(clean_df)
    forecast_model = train_demand_forecast_model(clean_df)

    # Simulate routing for 6 retail stores from a central depot
    depot = np.array([[0.0, 0.0]])
    stores = np.array([[2.5, 4.0], [1.2, 3.8], [6.0, 1.5], [5.5, 2.0], [3.0, 7.2], [6.2, 1.8]])
    store_demands = [180, 150, 210, 190, 160, 140]

    optimal_routes = optimize_delivery_route(depot, stores, store_demands, truck_capacity=500)
    print("\n--- OPTIMIZED REEFER DISPATCH ROUTES ---")
    for idx, (r, load) in enumerate(optimal_routes, 1):
        print(f"Truck {idx}: Sequence {r} | Total Payload: {load}/500 units")

--- BASELINE LOGISTICS KPI SUMMARY ---
           total_orders  otif_rate_pct  avg_lead_time_hrs  spoilage_rate_pct
rfc_id                                                                      
RFC_EAST            314          23.25              22.67               2.51
RFC_NORTH           338          30.18              22.00               2.55
RFC_SOUTH           348          25.29              22.13               2.29

Forecast Model Test MAPE: 35.75%

--- OPTIMIZED REEFER DISPATCH ROUTES ---
Truck 1: Sequence [0, 2, 1, 5, 0] | Total Payload: 490/500 units
Truck 2: Sequence [0, 4, 3, 0] | Total Payload: 400/500 units
Truck 3: Sequence [0, 6, 0] | Total Payload: 140/500 units
